**Purpose:** Exploratory analysis of GDELT v2 GKG data used to design the news filters.

**Notes:** paths resolve via `src.config` (run `pip install -e .` from the repo root first).


In [ ]:
from src.config import PROJECT_ROOT


In [1]:
import json

with open("news/gdelt_eda/gkg_eda_by_source.json", "r") as f:
    gkg_data = json.load(f)

In [2]:
import plotly.express as px

data = gkg_data  # your huge dataset

sectors = data[next(iter(data))].keys()

THRESHOLD = 0.0075  # 1%

for sector in sectors:
    # compute total for this sector across all sources
    total_sector_val = sum(vals[sector] for vals in data.values())

    labels = []
    sizes = []

    other_sum = 0

    for source, vals in data.items():
        val = vals[sector]
        if total_sector_val == 0:
            pct = 0
        else:
            pct = val / total_sector_val

        if pct < THRESHOLD:
            other_sum += val
        else:
            labels.append(source)
            sizes.append(val)

    if other_sum > 0:
        labels.append("Other (<1%)")
        sizes.append(other_sum)

    fig = px.pie(
        names=labels,
        values=sizes,
        title=f"{sector} distribution by source (filtered <.75%)"
    )
    fig.show()


In [3]:
# gkg_data is assumed to be a dict in the form:
# {
#    "source1": {"total": X, "Energy": Y, "Materials": Z, ...},
#    "source2": {...},
#    ...
# }

def compute_until_10pct(gkg_data):
    # Get list of sectors (all keys except "total")
    sample_source = next(iter(gkg_data.values()))
    sectors = [s for s in sample_source.keys() if s != "total"]

    results = {}

    for sector in sectors:
        # Collect all values for this sector
        sector_values = {
            source: vals.get(sector, 0)
            for source, vals in gkg_data.items()
        }

        # Total for the entire sector
        sector_sum = sum(sector_values.values())

        # Sort by sector value (descending)
        sorted_sources = sorted(sector_values.items(), key=lambda x: x[1], reverse=True)

        # Build cumulative table
        table = []
        cumulative_pct = 0.0

        for source, value in sorted_sources:
            if sector_sum > 0:
                pct_of_sector = value / sector_sum * 100
            else:
                pct_of_sector = 0

            pct_of_source = (value / gkg_data[source]["total"] * 100
                             if gkg_data[source]["total"] > 0 else 0)

            table.append({
                "source": source,
                "% of sector": pct_of_sector,
                "% of source": pct_of_source,
                "cumulative % of sector": cumulative_pct + pct_of_sector
            })

            cumulative_pct += pct_of_sector

            # Stop when cumulative percentage reaches 10%
            if cumulative_pct >= 10:
                break

        results[sector] = table

    return results

results = compute_until_10pct(gkg_data)

for sector, rows in results.items():
    print(f"\n=== Sector: {sector} ===")
    print(f"{'Source':40s} | {'% of sector':>12s} | {'% of source':>12s} | {'cumulative %':>12s}")
    print("-" * 85)
    for row in rows:
        print(f"{row['source']:40s} | {row['% of sector']:12.2f} | "
              f"{row['% of source']:12.2f} | {row['cumulative % of sector']:12.2f}")


=== Sector: Communication Services ===
Source                                   |  % of sector |  % of source | cumulative %
-------------------------------------------------------------------------------------
yahoo.com                                |         4.83 |         9.08 |         4.83
iheart.com                               |         2.30 |         8.90 |         7.13
msn.com                                  |         1.35 |         5.73 |         8.48
prnewswire.com                           |         1.00 |         7.80 |         9.48
gwdtoday.com                             |         0.73 |        12.67 |        10.21

=== Sector: Consumer Discretionary ===
Source                                   |  % of sector |  % of source | cumulative %
-------------------------------------------------------------------------------------
yahoo.com                                |         3.09 |         6.74 |         3.09
iheart.com                               |         1.72 |   

demorou 36h a correr

In [ ]:
# import json

# with open(str(PROJECT_ROOT / "01_data/processed/news_0.json"), "r") as f:
#     news_settings = json.load(f)
# news_settings["SourceCommonName_filter"] = {sector: [row["source"] for row in values] for sector, values in results.items()}
# with open(str(PROJECT_ROOT / "01_data/processed/news_0.json"), "w") as f:
#     json.dump(news_settings, f, indent=2)